In [2]:
import mysql.connector
import csv

DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "root",
    "database": "cooking_db",
    "charset": "utf8mb4"
}

conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()

with open("C:/Users/alstj/Desktop/GP/보관_권장_기간_원래_보존_최종.csv", encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    updated = 0
    skipped = 0

    for row in reader:
        name = row['품목'].strip()
        days_raw = row['최소 보관 권장 기간 (일)'].strip()

        # 숫자가 아닌 값(장기 보관 등)은 NULL 처리
        try:
            shelf_life = int(days_raw)
        except ValueError:
            shelf_life = None

        cursor.execute("""
            UPDATE ingredients
            SET shelf_life_days = %s
            WHERE name = %s
        """, (shelf_life, name))

        if cursor.rowcount > 0:
            updated += 1
        else:
            skipped += 1

conn.commit()
print(f"✅ 업데이트: {updated}개")
print(f"⚠️ 매칭 안됨: {skipped}개 (DB에 없는 재료)")

cursor.close()
conn.close()

✅ 업데이트: 268개
⚠️ 매칭 안됨: 158개 (DB에 없는 재료)
